# PCB Defect Detection Pipeline
This notebook documents an end-to-end pipeline for PCB defect detection using raw YOLO-format labels and image files. The code is organized to support data loading, visualization, preprocessing, scratch CNN training, transfer learning, evaluation, and project structure for submission.

## 1. Import Libraries and Set Paths
Import the necessary packages and define dataset directories. This cell is the starting point for all subsequent steps.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import seaborn as sns

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == 'notebooks':
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data_loader import build_annotation_manifest, CLASS_NAMES, get_class_counts, save_manifest_csv
from src.preprocess import create_tf_dataset, build_datasets, crop_and_resize, augment_image
from src.model import build_scratch_cnn, build_transfer_model

DATA_DIR = ROOT_DIR / 'dataset' / 'pcb-defect-dataset'
OUTPUT_DIR = ROOT_DIR / 'results'
MODEL_DIR = ROOT_DIR / 'models'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT_DIR)
print('Dataset:', DATA_DIR)
print('Class names:', CLASS_NAMES)

## 2. Load PCB Images and YOLO Labels
Read the raw YOLO annotation files and map each label to its source image. This pipeline extracts defect patches from the annotation manifest.

In [ ]:
train_manifest = build_annotation_manifest('train')
val_manifest = build_annotation_manifest('val')
test_manifest = build_annotation_manifest('test')

print('Train records:', len(train_manifest))
print('Val records:', len(val_manifest))
print('Test records:', len(test_manifest))

print('Train class distribution:')
print(get_class_counts(train_manifest))

print('Validation class distribution:')
print(get_class_counts(val_manifest))

print('Test class distribution:')
print(get_class_counts(test_manifest))


## 3. Visualize Images with Bounding Boxes
Verify the dataset by drawing YOLO bounding boxes and class labels on sample images.

In [ ]:
import cv2

def draw_boxes(image_path, labels):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    for class_id, x_center, y_center, width, height in labels:
        x_center *= w
        y_center *= h
        width *= w
        height *= h
        x1 = int(round(x_center - width / 2))
        y1 = int(round(y_center - height / 2))
        x2 = int(round(x_center + width / 2))
        y2 = int(round(y_center + height / 2))
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, CLASS_NAMES[class_id], (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

sample = train_manifest[0]
sample_labels = []
label_path = DATA_DIR / 'train' / 'labels' / sample['label_file']
with open(label_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split()
            sample_labels.append((int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])))
draw_boxes(sample['image_path'], sample_labels)

## 4. Preprocess Data: Resize and Normalize
Use the YOLO annotations to crop defect patches, resize them to a model input shape, and normalize pixel values.

In [ ]:
display_size = (224, 224)
sample_image = cv2.imread(str(sample['image_path']))
sample_patch = crop_and_resize(sample_image, sample['bbox'], display_size)

plt.figure(figsize=(4, 4))
plt.imshow(sample_patch)
plt.title(f'Cropped patch - {CLASS_NAMES[sample["class_id"]]}')
plt.axis('off')
plt.show()

## 5. Augmentation Pipeline for PCB Defect Images
Define augmentation transforms to improve generalization and make the model robust to noise and small variations.

In [ ]:
image_tensor = tf.convert_to_tensor(sample_patch, dtype=tf.float32)
image_tensor = tf.expand_dims(image_tensor, axis=0)
augmented = augment_image(image_tensor[0])

plt.figure(figsize=(4, 4))
plt.imshow(augmented.numpy())
plt.title('Augmented defect patch')
plt.axis('off')
plt.show()

## 6. Build a CNN Model from Scratch
Create a compact custom CNN architecture that can classify defect patches using only the raw dataset.

In [ ]:
scratch_model = build_scratch_cnn(input_shape=(*display_size, 3), num_classes=len(CLASS_NAMES))
scratch_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
scratch_model.summary()

## 7. Build a Transfer Learning Model
Load a pretrained MobileNetV2 backbone and adapt it for the PCB defect classes. This is the transfer learning path.

In [ ]:
transfer_model = build_transfer_model(input_shape=(*display_size, 3), num_classes=len(CLASS_NAMES), backbone='MobileNetV2')
transfer_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
transfer_model.summary()

## 8. Train Models and Record Metrics
Train both the scratch model and the transfer learning model using the same dataset preparation and compare their histories.

In [ ]:
train_ds, val_ds, test_ds = build_datasets(image_size=display_size, batch_size=32, augment=True)

history_scratch = scratch_model.fit(train_ds, validation_data=val_ds, epochs=8)

history_transfer_head = transfer_model.fit(train_ds, validation_data=val_ds, epochs=8)

# Fine-tune the top portion of the pretrained backbone to adapt the model to PCB defect textures.
for layer in transfer_model.backbone.layers[:100]:
    layer.trainable = False
for layer in transfer_model.backbone.layers[100:]:
    layer.trainable = True

transfer_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history_transfer_finetune = transfer_model.fit(train_ds, validation_data=val_ds, epochs=4)

## 9. Evaluate with Confusion Matrix and Training Graphs
Generate visual diagnostics that include confusion matrices and training curves for each model.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

def plot_history(history, title):
    plt.figure(figsize=(10, 4))
    plt.plot(history.history['accuracy'], label='train_accuracy')
    plt.plot(history.history['val_accuracy'], label='val_accuracy')
    plt.title(f'{title} Accuracy')
    plt.legend()
    plt.show()
    plt.figure(figsize=(10, 4))
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title(f'{title} Loss')
    plt.legend()
    plt.show()

plot_history(history_scratch, 'Scratch CNN')
plot_history(history_transfer_head, 'Transfer Learning Head Training')
plot_history(history_transfer_finetune, 'Transfer Learning Fine-Tuning')

def evaluate_model(model, dataset):
    y_true = []
    y_pred = []
    for x_batch, y_batch in dataset:
        preds = model.predict(x_batch, verbose=0)
        y_true.extend(np.argmax(y_batch.numpy(), axis=-1).tolist())
        y_pred.extend(np.argmax(preds, axis=-1).tolist())
    report = classification_report(y_true, y_pred, target_names=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)])
    cm = confusion_matrix(y_true, y_pred, labels=sorted(CLASS_NAMES))
    return report, cm

report_scratch, cm_scratch = evaluate_model(scratch_model, test_ds)
report_transfer, cm_transfer = evaluate_model(transfer_model, test_ds)

print('Scratch model report:\n', report_scratch)
print('Transfer model report (after fine-tuning):\n', report_transfer)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_scratch, annot=True, fmt='d', cmap='Blues', xticklabels=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)], yticklabels=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)])
plt.title('Scratch CNN Confusion Matrix')
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(cm_transfer, annot=True, fmt='d', cmap='Blues', xticklabels=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)], yticklabels=[CLASS_NAMES[i] for i in sorted(CLASS_NAMES)])
plt.title('Transfer Learning Confusion Matrix')
plt.show()


## 10. Compare Accuracy, Precision, Recall, F1
Summarize the classification metrics for both models and identify which one is stronger for PCB defect detection.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def summary_metrics(model, dataset):
    y_true = []
    y_pred = []
    for x_batch, y_batch in dataset:
        preds = model.predict(x_batch, verbose=0)
        y_true.extend(np.argmax(y_batch.numpy(), axis=-1).tolist())
        y_pred.extend(np.argmax(preds, axis=-1).tolist())
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    return precision, recall, f1

metrics_scratch = summary_metrics(scratch_model, test_ds)
metrics_transfer = summary_metrics(transfer_model, test_ds)

print('Scratch CNN - precision, recall, f1:', metrics_scratch)
print('Transfer learning - precision, recall, f1:', metrics_transfer)

## 11. Organize Clean GitHub Project Structure
This repository structure keeps the final submission clear and explainable. Use the scripts and notebook together to describe your workflow in the viva.

- `dataset/` — raw images and YOLO labels
- `src/` — `data_loader`, `preprocess`, `model`, `train`, `yolo_visualization` (main codebase)
- `scripts/` — thin wrappers that import `src` (backward compatible)
- `notebooks/` — viva walkthrough
- `results/` — plots, confusion matrices, metrics (created by `python -m src.train ...`)
- `app_streamlit.py` — optional demo UI
- `README.md` — commands and structure

Use the notebook during your viva to describe each stage: raw data parsing, annotation verification, patch-level preprocessing, model design, transfer learning, and final metric comparison.